<a href="https://colab.research.google.com/github/AliAnusKhan/Assignement1_Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliAnusKhan/Assignement1_Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Lane: CTR / Engagement Opportunity Scoring

Task type: This is a scoring/ranking problem. Given a content piece, we want to produce a continuous opportunity score that ranks pages by how much untapped CTR potential they have — not a binary classification, and not clustering into groups. The output is an ordered list of pages by opportunity, adjusted for their current position and volume.

## 2. Target or proxy

Since there is no direct "opportunity" label in the data, I will use a proxy target: the gap between a page's actual CTR and the expected CTR for its ranking position. Pages performing below the expected CTR for their position (avg_position) represent higher opportunity. This proxy is computed as:

opportunity_score = expected_ctr(avg_position) - actual_ctr

A higher positive value means bigger opportunity (underperforming relative to position).

## 3. Success metric
The primary success metric is ranking quality — specifically, whether pages flagged as high-opportunity actually show CTR improvement after being refreshed (measured via precision@k, i.e., of the top-K flagged pages, how many see a measurable CTR lift after action). A secondary metric is correlation between predicted opportunity score and actual subsequent CTR change (trend_pct), since the dataset includes historical trend data.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/AliAnusKhan/Assignement1_Flyrank/main/data/raw/content_refresh_anonymized.csv"
fly_rank= pd.read_csv(url)
print("Shape:", fly_rank.shape)
fly_rank[['content_id', 'client_id', 'avg_position', 'ctr', 'impressions_90d', 'clicks_90d']].head()


Shape: (30000, 44)


,content_id,client_id,avg_position,ctr,impressions_90d,clicks_90d
0,content_304f48230142,client_f369cb89fc,10.6,0.76,3803,29
1,content_a1fb4e703a9e,client_4e07408562,20.3,0.05,15320,7
2,content_9aa793d4d895,client_7f2253d7e2,36.5,0.09,12581,11
3,content_331d6c4de07b,client_19581e27de,6.2,0.49,11751,58
4,content_d99b7a2d90ca,client_3fdba35f04,44.0,0.13,19140,24


One row in this dataset represents one piece of content, for one client (identified by content_id + client_id). Each row carries that content's search performance metrics — position, CTR, impressions, clicks — over a 90-day window.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [7]:
import numpy as np

# Quantile-based binning instead of equal-width
fly_rank['position_bin'] = pd.qcut(fly_rank['avg_position'], q=10, duplicates='drop')
position_avg_ctr = fly_rank.groupby('position_bin')['ctr'].transform('mean')
fly_rank['opportunity_score'] = position_avg_ctr - fly_rank['ctr']

fly_rank[['content_id', 'avg_position', 'ctr', 'impressions_90d', 'opportunity_score']].sort_values(
    ['opportunity_score', 'impressions_90d'], ascending=[False, False]).head(10)

/tmp/ipykernel_1868/3045737869.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  position_avg_ctr = fly_rank.groupby('position_bin')['ctr'].transform('mean')


,content_id,avg_position,ctr,impressions_90d,opportunity_score
29080,content_50dfd64f9e8e,2.4,0.0,4446,1.491496
20777,content_134631e65b9e,1.5,0.0,4417,1.491496
7586,content_ce8619672faf,3.5,0.0,3855,1.491496
23829,content_747870c01e28,3.2,0.0,3585,1.491496
14987,content_cab2a611a390,3.4,0.0,3160,1.491496
9629,content_c90bfc85694f,1.1,0.0,3047,1.491496
13213,content_f2508318df4b,3.1,0.0,2831,1.491496
27660,content_856d5eb90281,3.6,0.0,2591,1.491496
23528,content_dce9528e18b1,3.1,0.0,2519,1.491496
5488,content_35d0f98ef4dc,1.8,0.0,2146,1.491496


A simple fixed rule (e.g., "flag anything below 2% CTR") would treat all low-CTR pages the same, regardless of context. This scoring approach instead surfaces a striking, specific pattern: content ranking in the top 4 search positions, receiving thousands of impressions (2,000–4,400+), yet earning zero clicks. This combination — high visibility, zero engagement — signals a likely title or meta-description mismatch, not a ranking problem. A fixed rule based on CTR alone couldn't isolate this pattern without also factoring in position and impression volume together, which is exactly what this position-adjusted scoring approach does. This ties directly to a real action: these specific pages should be prioritized immediately for title/meta rewrite, since they already have the visibility — they just need the click.

## Self-check

Before you submit, confirm each line honestly:

* [ ] Named the ML task type clearly — scoring/ranking (`position-adjusted opportunity score`)
* [ ] Defined a clear target/proxy — gap between actual CTR and expected CTR for a given position
* [ ] Defined a success metric — `precision@k` on flagged pages showing CTR lift after action; correlation with `trend_pct`
* [ ] Showed the unit of analysis as a real dataframe — one row = one piece of content, per client
* [ ] Explained why ML/adjusted scoring beats a fixed rule — surfaced top-ranked, high-impression, zero-CTR pages that a flat CTR threshold alone would miss
* [ ] Tied the output to a real content action — prioritize these pages for title/meta rewrite, since visibility already exists